In [ ]:
# you modify the following in the command line interface, like

### notebook config ###

table_path = None # where your metadata is (necessary)
output_folder = None # where to store your results
model_path = None # where to store your model
model_name = None # like 'efficientnet_b0' or 'mobilenet_v3_small'
num_epochs = None
batch_size = None
num_workers = None # speeds up audio processing
freeze_inner = None # true means only update output layer weights
head_only = None # true means only update the efficientnet or mobilenet heads
loss_name = None # 'cross_entropy' or 'bce_with_logits'
optimizer_name = None # 'adam' or 'sgd'
use_cosine_annealing = None # whether to use cosine annealing learning rate schedule
dropout = None # dropout rate for classifier head
lr = None # learning rate for optimizer
conduct_cv = None # cross-validation for model dev

### AudioDataset override knobs ###

base_folder = None # where high-level acoustics data files lie
duration = None # seconds in clip
p_augment = None # probability of training data augmentation
mixup_mean = None # mean of Poisson for mixup
num_augments = None # number of augmentation loops
mel_time_size = None # target time dimension size for mel spectrograms (applies random padding if specified)
label_smoothing_alpha = None # smoothing alpha for one-hot labels (BCE multi-label)

### Spectrogram augmentation knobs ###

max_gain = None
max_shift_pct = None
max_time_mask_pct = None
max_time_mask_num = None
max_freq_mask_len = None
max_freq_mask_num = None

p_gain = None
p_shift = None
p_time_mask = None
p_freq_mask = None

### don't change these ###

n_samples = None # downsample for software dev purposes
sr = None # standardized sampling rate

### Process the parameters

In [ ]:
# defaults
if duration is None:
    from osprey import duration
if base_folder is None:
    from osprey import base_folder
if sr is None:
    from osprey import sr
if p_augment is None:
    from osprey import p_augment
if num_augments is None:
    num_augments = 1
assert num_augments >= 1
num_augments = int(num_augments)

# spectrogram augmentation settings
if max_gain is None:
    max_gain = 15.
if max_shift_pct is None:
    max_shift_pct = 0.04
if max_time_mask_pct is None:
    max_time_mask_pct = 0.02
if max_time_mask_num is None:
    max_time_mask_num = 4
if max_freq_mask_len is None:
    max_freq_mask_len = 3
if max_freq_mask_num is None:
    max_freq_mask_num = 4

if p_gain is None:
    p_gain = p_augment
if p_shift is None:
    p_shift = p_augment
if p_time_mask is None:
    p_time_mask = p_augment
if p_freq_mask is None:
    p_freq_mask = p_augment

if mixup_mean is None:
    mixup_mean = 0.5

# model architecture
if model_name is None:
    model_name = 'efficientnet_b0'
if freeze_inner is None:
    freeze_inner = False
if head_only is None:
    head_only = True

# training details
if conduct_cv is None:
    conduct_cv = True
if num_epochs is None:
    num_epochs = 5
if batch_size is None:
    batch_size = 2 ** 8
if loss_name is None:
    loss_name = 'bce'
if optimizer_name is None:
    optimizer_name = 'adam'
if use_cosine_annealing is None:
    use_cosine_annealing = False
if lr is None:
    lr = 4e-4
if num_workers is None:
    num_workers = 0
if n_samples is None: # to study small subsamples
    n_samples = 1e9

# paths to folders and files
collection_map = {
    'iNat': 'birdclef-2026/train_audio',
    'XC': 'birdclef-2026/train_audio',
    'iNat2': 'birdclef-2025/train_audio',
    'XC2': 'birdclef-2025/train_audio',
    'CSA': 'birdclef-2025/train_audio',
    'esc': 'ESC-50-master/audio',
    'arca23k': 'ARCA23K/ARCA23K.audio/ARCA23K.audio',
    'urban8k': 'UrbanSound8K/audio/foldall',
    'dbr': 'dbr-dataset/dataset/dograin',
    'freesound': 'freesound/outside',
    'random-noise': 'random-noise',
}

### HANDLE SOUNDSCAPES ###

if output_folder is None:
    output_folder = '../results'
loss_folder = f"{output_folder}/loss"
metrics_folder = f"{output_folder}/metrics"

# trigger cli failure
assert table_path is not None, 'Must provide an audio metadata file'

### Importing packages and other setup

In [ ]:
# common imports
import os
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ( 
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
)


# modeling imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models
from safetensors.torch import save_file, load_file, safe_open
import json

# osprey package
from osprey import (
    SpectrogramDataset,
    FocalBCEWithLogitsLoss,
    reformat_image,
    augmenter_spectrogram,
    )

def apply_spectrogram_augmentation(
    x,
    *,
    p_gain,
    p_time_mask,
    p_freq_mask,
 ):
    return augmenter_spectrogram(
        x,
        max_gain=max_gain,
        p_gain=p_gain,
        max_time_mask_pct=max_time_mask_pct,
        max_time_mask_num=max_time_mask_num,
        p_time_mask=p_time_mask,
        max_freq_mask_len=max_freq_mask_len,
        max_freq_mask_num=max_freq_mask_num,
        p_freq_mask=p_freq_mask,
    )

os.makedirs(output_folder, exist_ok=True)
os.makedirs(loss_folder, exist_ok=True)
os.makedirs(metrics_folder, exist_ok=True)

# Check if a GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU instead.")

### Loading and preparing the data

In [ ]:
# table with metadata about audio files
meta = pd.read_csv(table_path)

# subsets
if conduct_cv:
    train_data = meta[meta['dataset']=='train'].copy()
    le = LabelEncoder() # label encoder
    le.fit(train_data['primary_label'])
    val_data = meta[meta['dataset'].isin(['validate','test'])].copy()
else:
    train_data = meta[meta['dataset'].isin(['train','validate','test'])].copy()
    le = LabelEncoder() # label encoder
    le.fit(train_data['primary_label'])
    val_data = meta[meta['dataset'].isin([])].copy()


# reset indices
train_data.reset_index(inplace=True, drop=True)
val_data.reset_index(inplace=True, drop=True)

# unique species in datasets
train_species = np.array(train_data.primary_label.unique())
val_species = np.array(val_data.primary_label.unique())

# smaller subset: 8 batches of 32 samples = 256
num_classes = len(le.classes_)

# Determine label encoding based on loss function
# BCE-based losses (bce, focal_bce) require one-hot encoding
# CrossEntropy-based losses (cross_entropy, focal_cross_entropy) require class indices
encode_labels_onehot = loss_name in ['bce', 'focal_bce']

# sample from test split (use .head(n_samples) instead of .sample(...) if you want deterministic order)
train_data_small = train_data.sample(
    n=min(n_samples, len(train_data)), 
    random_state=42).reset_index(drop=True)
val_data_small = val_data.sample(
    n=min(n_samples, len(val_data)), 
    random_state=42).reset_index(drop=True)

# optional: overwrite active dataset/dataloader to use the small subset
dataset_train = SpectrogramDataset(
    train_data_small,
    le,
    base_folder=base_folder,
    collection_map=collection_map,
    encode_labels_onehot=encode_labels_onehot,
    mel_time_size=mel_time_size,
    label_smoothing_alpha=(label_smoothing_alpha if label_smoothing_alpha is not None else 0.0),
 )
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, num_workers=num_workers, shuffle=True)
dataset_val = SpectrogramDataset(
    val_data_small,
    le,
    base_folder=base_folder,
    collection_map=collection_map,
    encode_labels_onehot=encode_labels_onehot,
    mel_time_size=mel_time_size,
    label_smoothing_alpha=(label_smoothing_alpha if label_smoothing_alpha is not None else 0.0),
)
dataloader_val = DataLoader(dataset_val, batch_size=batch_size, num_workers=num_workers, shuffle=True)

print("Number of training samples:", len(train_data_small))
print("Number of batches:", len(dataloader_train))

### Model definition

In [ ]:
weights_map = {
    'efficientnet_b0': models.EfficientNet_B0_Weights.DEFAULT,
    'efficientnet_b1': models.EfficientNet_B1_Weights.DEFAULT,
    'efficientnet_b2': models.EfficientNet_B2_Weights.DEFAULT,
    'efficientnet_b3': models.EfficientNet_B3_Weights.DEFAULT,
    'mobilenet_v3_small': models.MobileNet_V3_Small_Weights.DEFAULT,
    'mobilenet_v3_large': models.MobileNet_V3_Large_Weights.DEFAULT,
}
weights = weights_map[model_name]
model_fn = getattr(models, model_name)
model = model_fn(weights=weights)

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)

# total parameter counts
total_params = sum(p.numel() for p in model.parameters())

if freeze_inner:

    # freeze all but the last layer
    for param in model.parameters():
        param.requires_grad = False
    # the head
    if head_only:
        for param in model.classifier[-1].parameters():
            param.requires_grad = True
    else:
        for param in model.classifier.parameters():
            param.requires_grad = True


# trainable parameter counts
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

model.to(device)

# printing summary
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {total_params - trainable_params:,}")

### Optimizer and loss function

In [ ]:
if loss_name == 'bce':
    criterion = nn.BCEWithLogitsLoss()
elif loss_name == 'focal_bce':
    criterion = FocalBCEWithLogitsLoss(alpha=0.25, gamma=2.0)
else:
    raise ValueError(f"Unknown loss_name: {loss_name}")

if optimizer_name == 'adam':
    optimizer = optim.Adam(model.parameters(), lr=lr)
elif optimizer_name == 'adamw':
    optimizer = optim.AdamW(model.parameters(), lr=lr)
else:
    raise ValueError(f"Unknown optimizer_name: {optimizer_name}")

scheduler = None
if use_cosine_annealing:
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

### Training loop

In [ ]:
ft = open(f"{loss_folder}/training_loss.csv", 'w')
fv = open(f"{loss_folder}/validation_loss.csv", 'w')
ft.write('epoch,loss\n')
fv.write('epoch,loss\n')

for _ in range(1, num_epochs + 1):

    # learning
    model.train()
    loss_total = 0
    itr = 0
    for a in range(1, num_augments + 1):
        curr_time = time.time()
        itr = 0
        for X, Y in dataloader_train:
            itr += 1
            if (itr % 200) == 0:
                print(f'{itr/len(dataloader_train)*100:.4f}%')
            optimizer.zero_grad()
            X = X.to(device)
            ### HANDLE MIXUP ###
            X = apply_spectrogram_augmentation(
                X,
                p_gain=p_gain,
                p_time_mask=p_time_mask,
                p_freq_mask=p_freq_mask,
            )
            Z = reformat_image(X,)
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss.item()
            loss.backward()
            optimizer.step()
        print(f'Epoch {_}, Round {a} took {time.time() - curr_time:0f} seconds')
    print('')

    # inference
    model.eval()
    with torch.no_grad():

        # training loss tracking
        loss_total = 0
        itr = 0
        for X, Y in dataloader_train:
            itr += 1
            X = X.to(device)
            Z = reformat_image(X,)
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss.item()
        print(f'The training loss at epoch {_} is {loss_total / itr:6f}')
        ft.write(f"{_},{loss_total / itr:4f}\n")

        # validation loss tracking
        loss_total = 0
        itr = 0
        for X, Y in dataloader_val:
            itr += 1
            X = X.to(device)
            Z = reformat_image(X,)
            Y = Y.to(device)
            logits = model(Z)
            loss = criterion(logits, Y)
            loss_total += loss.item()
        print(f'The validation loss at epoch {_} is {loss_total / itr:6f}')
        fv.write(f"{_},{loss_total / itr:4f}\n")
    
    print('')

ft.close()
fv.close()

### Save model

In [ ]:
if model_path is not None:

    checkpoint_path = f"{output_folder}/{model_path}"

    tensor_state_dict = model.state_dict()

    metadata = {
        "optimizer_state_dict": json.dumps(optimizer.state_dict(), default=str),
        "num_classes": str(num_classes),
        "label_classes": json.dumps(le.classes_.tolist()),
        "model_name": model_name,
        "loss_name": str(loss_name),
        "epoch": str(num_epochs),
        "num_augments": str(num_augments),
        "p_augment": str(p_augment),
        "mixup_mean": str(mixup_mean),
    }

    save_file(tensor_state_dict, checkpoint_path, metadata=metadata)

    print(f"Checkpoint saved safely to: {checkpoint_path}")

### Load the saved model

In [ ]:
weights_map = {
    'efficientnet_b0': models.EfficientNet_B0_Weights.DEFAULT,
    'efficientnet_b1': models.EfficientNet_B1_Weights.DEFAULT,
    'efficientnet_b2': models.EfficientNet_B2_Weights.DEFAULT,
    'efficientnet_b3': models.EfficientNet_B3_Weights.DEFAULT,
    'mobilenet_v3_small': models.MobileNet_V3_Small_Weights.DEFAULT,
    'mobilenet_v3_large': models.MobileNet_V3_Large_Weights.DEFAULT,
}

with safe_open(checkpoint_path, framework='pt') as f:
    m = f.metadata()
    num_classes = int(m['num_classes'])
    model_name = m['model_name']

weights = weights_map[model_name]
model_fn = getattr(models, model_name)
model = model_fn(weights=weights)

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)

state_dict = load_file(checkpoint_path)
model.load_state_dict(state_dict)

model.to(device)

print('Loaded the model successfully')

### Evaluation loop

In [ ]:
# compute some summary metrics

zipped_data = [
    (train_species, dataloader_train, 'training'),
    (val_species, dataloader_val, 'validation'),
]

for species, dataloader, split_name in zipped_data:

    g = open(f"{metrics_folder}/{split_name}_metrics.csv",'w')

    model.eval()

    # Initialize empty lists to hold batch tensors
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for X, Y in dataloader:
            # Move everything to device immediately
            X = X.to(device)
            Z = reformat_image(X,)
            Y = Y.to(device) 
            
            logits = model(Z)

            # Store the batch tensors directly
            y_true_all.append(Y)
            y_pred_all.append(logits)

    # Concatenate all batches at once and move to CPU only once
    y_true = torch.cat(y_true_all).cpu().numpy()
    y_logits = torch.cat(y_pred_all).cpu().numpy()

    # Metrics depend on loss function
    # BCE multilabel: y_true is one-hot, calculate AUROC, AP, precision and recall at two thresholds per label
    g.write('label,auroc,ap,precision_0.5,precision_0.8,recall_0.5,recall_0.8\n')

    y_true_binary = (y_true > 0.5).astype(int)
    
    # Calculate probability predictions
    y_probs = 1 / (1 + np.exp(-y_logits))  # sigmoid(logits)
    
    # Per-label metrics
    per_label_aurocs = []
    per_label_aps = []
    
    for label_idx in range(y_true.shape[1]):
        label_name = le.inverse_transform([label_idx])[0]
        auroc = np.nan
        ap = np.nan
        prec_05 = np.nan
        prec_08 = np.nan
        rec_05 = np.nan
        rec_08 = np.nan
        
        try:
            auroc = roc_auc_score(y_true_binary[:, label_idx], y_probs[:, label_idx])
            per_label_aurocs.append(auroc)
        except:
            pass
        
        try:
            ap = average_precision_score(y_true_binary[:, label_idx], y_probs[:, label_idx])
            per_label_aps.append(ap)
        except:
            pass
        
        # Precision and recall at threshold 0.5
        try:
            pred_05 = (y_probs[:, label_idx] >= 0.5).astype(int)
            prec_05 = precision_score(y_true_binary[:, label_idx], pred_05, zero_division=0)
            rec_05 = recall_score(y_true_binary[:, label_idx], pred_05, zero_division=0)
        except:
            pass
        
        # Precision and recall at threshold 0.8
        try:
            pred_08 = (y_probs[:, label_idx] >= 0.8).astype(int)
            prec_08 = precision_score(y_true_binary[:, label_idx], pred_08, zero_division=0)
            rec_08 = recall_score(y_true_binary[:, label_idx], pred_08, zero_division=0)
        except:
            pass
        
        g.write(f"{label_name},{auroc:.4f},{ap:.4f},{prec_05:.4f},{prec_08:.4f},{rec_05:.4f},{rec_08:.4f}\n")
    
    # Calculate summary statistics
    mean_ap = np.nanmean(per_label_aps) if per_label_aps else np.nan
    median_ap = np.nanmedian(per_label_aps) if per_label_aps else np.nan
    
    try:
        overall_auroc = roc_auc_score(y_true_binary, y_probs, average='macro')
    except:
        overall_auroc = np.nan
    
    g.write(f"mean_AP,nan,{mean_ap:.4f},nan,nan,nan,nan\n")
    g.write(f"median_AP,nan,{median_ap:.4f},nan,nan,nan,nan\n")
    g.write(f"overall_auroc,{overall_auroc:.4f},nan,nan,nan,nan,nan\n")

    g.close()